# 02 · Baselines + BERT fine-tuning on GPU

Needs the results of notebook 01 **committed and pushed** to the branch below
(`data/processed/cv_text.csv`, `data/splits/*.csv`, `data/raw/metadata.csv`).
No PaddlePaddle here — only PyTorch / Transformers.

**Before running:** Kaggle → *Accelerator GPU T4/P100, Internet On*; Colab → *T4 GPU*.
Total time on a T4: roughly 1–2 h including the Q3 variants.


In [ ]:
# ---- repository and working branch ----
REPO_URL = "https://github.com/karelysscs/cv_experience_ocr_bert.git"
BRANCH = "Practica_04"

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
BASE = "/kaggle/working" if IN_KAGGLE else "/content"
REPO = f"{BASE}/cv_experience_ocr_bert"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(["git", "pull"], check=False)
print("Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "other", "| cwd:", os.getcwd())
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# torch is preinstalled on Colab/Kaggle; do not reinstall it
!pip install -q -U "transformers>=5" "sentence-transformers>=6" accelerate jiwer
import torch, transformers, sentence_transformers
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("transformers", transformers.__version__, "| sentence-transformers", sentence_transformers.__version__)
assert os.path.exists("data/splits/train.csv"), "Commit + push the outputs of notebook 01 first"

## 1 · Baselines (main setup: cleaned OCR text)

In [ ]:
!python -m src.baseline_median
!python -m src.baseline_tfidf
!python -m src.baseline_rules
!python -m src.baseline_embeddings --encoder BAAI/bge-small-en-v1.5

## 2 · Fine-tune BERT (main setup)
DistilBERT, head truncation at 512 tokens, MSE, lr 5e-5, batch 16, ≤10 epochs with early stopping on validation MAE. Check `date_survival` in `output/metrics/runs.json`: if many CVs lose dates to truncation, prefer `--strategy head_tail` or `chunk` as the main setup.

In [ ]:
!python -m src.train_bert --model distilbert-base-uncased --strategy head --loss mse

In [ ]:
import json
r = json.load(open("output/metrics/runs.json"))["bert"]
print(json.dumps({k: r[k] for k in ["val_mae", "train_s", "infer_s_per_cv", "epochs_run", "hardware", "date_survival"]}, indent=2))

## 3 · Q3 sensitivity experiments
* long-text strategy: head vs head+tail vs chunking
* loss: MSE vs Huber
* TF-IDF numbers: kept vs removed
* OCR cleaning: raw vs cleaned vs low-confidence lines removed (every model)

In [ ]:
!python -m src.train_bert --strategy head_tail
!python -m src.train_bert --strategy chunk
!python -m src.train_bert --loss huber
!python -m src.baseline_tfidf --no-digits

In [ ]:
for text in ["ocr_raw", "ocr_conf"]:
    !python -m src.baseline_tfidf --train-text {text}
    !python -m src.baseline_embeddings --train-text {text}
    !python -m src.baseline_rules --train-text {text}
    !python -m src.train_bert --train-text {text}

## 4 · Probe (Q1), evaluation and figures

In [ ]:
!python -m src.probe
!python -m src.evaluation
!python -m src.visualization
import pandas as pd
display(pd.read_csv("output/tables/model_comparison.csv"))
display(pd.read_csv("output/tables/bert_gain_ci.csv"))

## 5 · Download
* `stage2_outputs.zip` → tables, figures, metrics, predictions: **commit these**.
* `stage2_models.zip` → baseline `.joblib` files + the main BERT model (`models/bert/bert`, ~250 MB) for the Streamlit estimator: **do not commit** (git-ignored); unzip them into your local repo.

In [ ]:
import glob, zipfile
!pip freeze | grep -i -E "^(torch|transformers|sentence-transformers|accelerate|scikit-learn|numpy|pandas)==" > output/metrics/env_models.txt
with zipfile.ZipFile(f"{BASE}/stage2_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("output/**/*", recursive=True):
        if os.path.isfile(f): z.write(f)
with zipfile.ZipFile(f"{BASE}/stage2_models.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("models/baselines/*.j*") + glob.glob("models/bert/bert/*"):
        if os.path.isfile(f): z.write(f)
for f in ["stage2_outputs.zip", "stage2_models.zip"]:
    print(f, os.path.getsize(f"{BASE}/{f}") // 1e6, "MB")
if IN_COLAB:
    from google.colab import files
    files.download(f"{BASE}/stage2_outputs.zip"); files.download(f"{BASE}/stage2_models.zip")